# ガイドカメラ(Atik314L+)で写真と撮る処理をCLI実行するプログラム検討のためのコード

## 1. 接続確認

In [10]:
import win32com.client
import time

def test_atik_final():
    # 先ほど特定した正確な ProgID
    prog_id = "ASCOM.AtikCameras.Camera" 
    
    try:
        print(f"Connecting to [{prog_id}]...")
        cam = win32com.client.Dispatch(prog_id)
        
        # 接続
        cam.Connected = True
        
        if cam.Connected:
            print("--- 🚀 接続成功 ---")
            print(f"モデル名: {cam.Name}")
            print(f"現在の温度: {cam.CCDTemperature} ℃")
            
            # 冷却を少しだけ動かしてみる
            print("冷却テストを開始します...")
            cam.CoolerOn = True
            time.sleep(2)
            print(f"現在の冷却パワー: {cam.CoolerPower} %")
            
            # 安全に切断
            cam.Connected = False
            print("--- 👋 正常に切断しました ---")
        else:
            print("接続に失敗しました。")

    except Exception as e:
        print(f"エラーが発生しました: {e}")

if __name__ == "__main__":
    test_atik_final()

Connecting to [ASCOM.AtikCameras.Camera]...
--- 🚀 接続成功 ---
モデル名: Atik Camera Driver
現在の温度: 20.18 ℃
冷却テストを開始します...
現在の冷却パワー: 0.0 %
--- 👋 正常に切断しました ---


## 2. テスト撮像

In [1]:
import os
from base__atik_camera_contloer import AtikCamera

In [ ]:
output_dir = '../pictures/02_slit_guide_images'

# 露光時間（秒）・目標CCD温度はここで変更可能
exposure_sec = 5.0
target_temp_c = -5.0  # 26.3.4昼時点のCCDは0.6度なので、簡易テスト用に-5度を設定
save_enabled = True  # 撮像結果を保存するかの二値
loop_capture_times = 5  # test用に5回撮像

cam = AtikCamera()

In [3]:
# 接続
cam.connect()
print("--- 接続完了 ---")
print(f"カメラ: {cam.camera.Name}")
print(f"解像度: {cam.camera.CameraXSize} x {cam.camera.CameraYSize}")

# 冷却: 目標温度まで待機
print(f"\n冷却開始: 目標 {target_temp_c} ℃")
cam.cool_to(target_temp_c)

# 撮像（ライトフレーム）
for i in range(loop_capture_times):
    print(f"\n露光開始: {exposure_sec} 秒")
    image = cam.capture(exposure_sec)
    print(f"画像取得完了: shape {image.shape}")

    # FITS 保存
    save_filename = f'test_{i:02d}'
    path = cam.save_fits(image, exposure_sec, output_dir=output_dir, filename_base=save_filename, enabled=save_enabled)
    if path is None:
        print("保存はスキップしました。")  # TODO: 動作確認後、skipに変更
    else:
        print(f"保存しました: {path}")

# 終了時は必ず切断
cam.disconnect()
print("\n--- 切断完了 ---")

--- 接続完了 ---
カメラ: Atik Camera Driver
解像度: 1391 x 1039

冷却開始: 目標 -5.0 ℃
  CCD温度: -4.95 ℃, 冷却パワー: 69.4 %
  目標温度 -5.0 ℃ 付近に到達しました。

露光開始: 5.0 秒
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: 4
  ステータス: 4
画像取得完了: shape (1039, 1391)
保存しました: ../pictures/02_slit_guide_camera\test_00.fits

露光開始: 5.0 秒
  ステータス: Waiting
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: 4
  ステータス: 4
画像取得完了: shape (1039, 1391)
保存しました: ../pictures/02_slit_guide_camera\test_01.fits

露光開始: 5.0 秒
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: Exposing
  ステータス: 4
  ステータス: 4
  ステータス: 4
画像取得完了: shape (1039, 1391)
保存しました: ..